# Feedforward

**Feedforward** is the mechanism by which the parameter of an operation depends on a measurement result
obtained earlier in the same execution. It is not a convenience: a measurement-based computer computes
*by* measuring, and the correction that turns a measurement outcome into the intended operation is
feedforward: a teleportation shifts the mode it outputs by an amount that depends on the measured
values, and feedforward is the correction that cancels that shift.

It comes in two pieces, and the difference between them decides what you can write:

* A **variable** stands for a value that will be measured. Passing one straight to an operation makes
  that parameter follow the measured value.
* A **function** is applied to a variable when the parameter should depend on the result in some other
  way. It crosses to the simulator or the machine as source code, so what it may contain is restricted.

Everything on this page runs on the simulator, so the outputs below are real.

In [ ]:
# Draw the figures on this page as SVG so that they stay sharp at any zoom level.
# The cell itself is hidden from the built page by the `remove-cell` tag.
%config InlineBackend.figure_formats = ["svg"]

## Variables

A variable represents a measurement result obtained during execution, as a
{py:class}`~claveles.feedforward.Variable` or a subclass of it. Where one comes from depends on the
representation:

| Representation | How a variable is obtained |
| --- | --- |
| Circuit | The measurement operation returns it, as the value of the `\|` expression that applied it. See {ref}`sec:circuit-adding-feedforward`. |
| Graph | {py:meth}`~claveles.graph.GraphRepr.get_measured_value` for a macronode and a micronode, or {py:meth}`~claveles.graph.GraphRepr.get_mode_measured_value` for a mode. See [Constructing Graph Representation](graph_repr.ipynb). |
| Machinery | {py:class}`~claveles.machinery.macronode_angle.MeasuredVariable`, naming a macronode and a micronode. See [Constructing Machinery Representation](machinery_repr.ipynb). |

Passing a variable to an operation on its own is the simplest case, and needs no function at all:

In [ ]:
from math import pi

from claveles.circuit import CircuitRepr
from claveles.circuit.ops import std
from claveles.circuit.state import QuantumState
from claveles.circuit.visualize import make_figure

circuit = CircuitRepr("variable")
circuit.Q(0) | QuantumState.squeezed(r=1.0, phi=0.0)
circuit.Q(1) | QuantumState.vacuum()

measured = circuit.Q(0) | std.MeasureHomodyne(0.0)
circuit.Q(1) | std.PhaseRotation(measured)  # rotate mode 1 by whatever mode 0 read

print(type(measured).__name__)

make_figure(circuit);

## Functions

A feedforward function is a {py:class}`~claveles.feedforward.FeedForwardFunction`, made by decorating an
ordinary Python function with {py:deco}`~claveles.feedforward.feedforward`. Applying it to a variable gives
the parameter to pass to an operation:

In [ ]:
from claveles.feedforward import feedforward


@feedforward
def half_atan(x: float) -> float:
    from math import atan

    return atan(x) / 2


circuit = CircuitRepr("function")
circuit.Q(0) | QuantumState.squeezed(r=1.0, phi=0.0)
circuit.Q(1) | QuantumState.vacuum()

m0 = circuit.Q(0) | std.MeasureHomodyne(0.0)
circuit.Q(1) | std.PhaseRotation(half_atan(m0))
m1 = circuit.Q(1) | std.MeasureHomodyne(pi / 2)

print(type(half_atan).__name__)

make_figure(circuit);

That circuit runs like any other. The rotation applied to mode 1 differs from shot to shot, because
what mode 0 read differs:

In [ ]:
from claveles.client import SimulatorClient
from claveles.execute import execute

result = execute(circuit, SimulatorClient(n_shots=3))
for shot in range(3):
    print(f"shot {shot}: {result[shot]}")

(sec:feedforward-functions-contents)=

### What a function may contain

The function is sent as source code and executed by whatever runs the circuit, so the check on its
contents is the only thing keeping it to arithmetic. **Anything not in the list below is refused**, and
a construct that went unrecognized would be a hole rather than an omission.

* The source must define exactly one function, taking a `float` and returning a `float`, with exactly
  one `return` statement.
* It must be self-contained, apart from declared parameters. A name from the surrounding scope may be
  read only when it holds an {py:class}`~claveles.feedforward.FFParameter`; see
  {ref}`sec:feedforward-parameters` below.
* Only the `math` module may be imported, only as `from math import foo`, and only **inside** the
  function.
* Permitted otherwise:
  * Assignments (`=`, `:=`) and augmented assignments (`+=`, `*=`, and the rest)
  * Unary operators `+`, `-`, `~`, `not`
  * Binary operators `+`, `-`, `*`, `/`, `//`, `%`, `<<`, `>>`, `&`, `^`, `|`, `**`, `@`
  * Comparisons `<`, `<=`, `>`, `>=`, `==`, `!=` and the boolean `and`, `or`
  * The conditional expression `a if c else b`. The statement form of `if` is not allowed, so a
    decision is written this way; `float(m > 0)` and `1.0 if m > 0 else 0.0` are both fine.
  * The `real` and `imag` parts of a number, for a function producing a complex gate parameter
  * Calls into the `math` module, and the built-ins `abs`, `bool`, `complex`, `divmod`, `float`, `int`,
    `pow`, `round`

Refused in particular: `is` and `in`, attribute access other than `real` and `imag`, subscripting, list,
tuple, dict and set literals, f-strings, lambdas, comprehensions and generator expressions. A name the
function assigned cannot be called, since it holds a value rather than a function.

The decorator checks all of this immediately, so a function that cannot be sent is refused where it is
written rather than at submission:

In [ ]:
try:

    @feedforward
    def subscripting(x: float) -> float:
        return [x, 2 * x][0]

except ValueError as error:
    print("ValueError:", error)

{py:func}`~claveles.feedforward.verification.verify_feedforward` runs the same check on a source string, and
{py:meth}`FeedForwardFunction.verify <claveles.feedforward.FeedForwardFunction.verify>` on a function that
has not been decorated yet. Both raise on a function that would be refused and return quietly otherwise,
which is what to reach for when the source is assembled rather than written out:

In [ ]:
from claveles.feedforward import FeedForwardFunction


def candidate(x: float) -> float:
    return 2 * x + 1


FeedForwardFunction.verify(candidate)  # raises if it could not be sent
print("accepted")

(sec:feedforward-parameters)=

## Parameters

A feedforward function runs with nothing in scope, so a name it reads has to become a literal before it
is sent. Declare such a name as a *parameter* with {py:func}`~claveles.feedforward.ff_parameters`, and give
it a value with {py:meth}`~claveles.feedforward.FFParameter.set`. {py:func}`~claveles.feedforward.ff_parameter`
declares a single one and returns it directly rather than in a tuple:

In [ ]:
from claveles.feedforward import ff_parameters

gain, offset = ff_parameters("gain", "offset")


@feedforward
def tuned(m: float) -> float:
    from math import atan

    return atan(gain * m) / (1 + offset * m**2)


gain.set(1.0)
offset.set(2.0)
print(gain, offset)

The value is read when the circuit is **serialized**, not when the function is defined. That is why
{py:meth}`~claveles.feedforward.FFParameter.set` may come after the function is written, and why changing a
value and submitting again sends the new one — handy when a notebook cell is re-run with a different
setting.

Declaring is required. A name that was not declared is refused even if it happens to hold a number, so a
typo cannot be embedded silently:

In [ ]:
gamma = 9.9  # an ordinary float, not a declared parameter

try:

    @feedforward
    def uses_gamma(m: float) -> float:
        return gamma * m

except ValueError as error:
    print("ValueError:", error)

A parameter may hold an `int`, a `float` or a `complex`, and a NumPy scalar is accepted and converted.
One left without a value raises a `ValueError` when the circuit is serialized, naming the parameter:

In [ ]:
never_set = ff_parameters("never_set")[0]


@feedforward
def uses_never_set(m: float) -> float:
    return never_set * m


circuit = CircuitRepr("unset_parameter")
circuit.Q(0) | QuantumState.vacuum()
circuit.Q(1) | QuantumState.vacuum()
m0 = circuit.Q(0) | std.MeasureHomodyne(0.0)
circuit.Q(1) | std.PhaseRotation(uses_never_set(m0))

try:
    circuit.proto()
except ValueError as error:
    print("ValueError:", error)

Because the value is resolved by Python's own scoping rules, a function that *builds* a parametrized
feedforward function works as expected — each call gets its own parameter:

```python
def make_ff(value):
    gain = ff_parameters("gain")[0]
    gain.set(value)

    @feedforward
    def f(m):
        return gain * m

    return f
```

## Feedforwarding a complex parameter

A gate whose parameter is a complex amplitude, such as
{py:class}`~claveles.circuit.ops.std.Displacement`, takes an ordinary complex expression over the measured
variables. No {py:deco}`~claveles.feedforward.feedforward` function is needed:

In [ ]:
circuit = CircuitRepr("complex_parameter")
for mode in range(3):
    circuit.Q(mode) | QuantumState.vacuum()

m0 = circuit.Q(0) | std.MeasureHomodyne(0.0)
m1 = circuit.Q(1) | std.MeasureHomodyne(pi / 2)
circuit.Q(2) | std.Displacement(m0 - 1j * m1)

print("operations =", circuit.n_operations)

make_figure(circuit);

The value such an expression builds is a {py:class}`~claveles.feedforward.ComplexFeedForward`, which holds
the two parts separately rather than as a Python `complex`, because each carries its own feedforward
function.

The real and imaginary parts of the amplitude drive the two quadratures independently, so **each part
may follow at most one measurement result**. That limits these expressions to affine ones: scaling by a
purely real or purely imaginary scalar, and adding a constant. Combining two measurement results into
one quadrature, or scaling by a scalar with both a real and an imaginary part, is refused:

In [ ]:
circuit = CircuitRepr("two_results_one_quadrature")
for mode in range(3):
    circuit.Q(mode) | QuantumState.vacuum()

m0 = circuit.Q(0) | std.MeasureHomodyne(0.0)
m1 = circuit.Q(1) | std.MeasureHomodyne(pi / 2)

try:
    circuit.Q(2) | std.Displacement(m0 + m1)
except ValueError as error:
    print("ValueError:", error)

A non-linear function of one result, such as `sin`, still requires an explicit feedforward function, as
{ref}`above <sec:feedforward-functions-contents>` describes. A feedforward function takes a single
measurement result, so it does not lift the limit: a product of two results, or any other combination of
them in one quadrature, cannot be fed forward at all.

## Where feedforward is executed

On a remote server, a feedforward function is executed under Ubuntu 22.04 or Ubuntu 24.04.

On the machine, feedforward is not applied to the quantum states at all: it is applied to the measured
values afterwards, which is equivalent for Gaussian states and is what the hardware actually does.